# Generating a Network's Weights with a Hypernetwork

Companion notebook for the [blog post](https://Notabot123.github.io/polyweave/blog/hypernetworks-on-cifar/).

This is a **self-contained API demo on random data** so it runs anywhere with no download.
The teacher is untrained and the data is random, so accuracies are near chance - the point
is to show the pieces fit together. For real CIFAR-10 numbers, run the experiment (note at
the end).

In [ ]:
import copy
import torch

from polyweave.students import make_cnn_student
from polyweave.prototypes import feature_class_stats
from polyweave.hypernets import FCMapTeacher
from polyweave.evaluation import (
    generate_averaged, evaluate_accuracy,
    class_centroids, centroids_to_fc, random_like,
    recovery_curve, ensemble_accuracy, ensemble_gain, pairwise_disagreement,
)

torch.manual_seed(0)
NUM_CLASSES, FEATURE_DIM = 10, 256

## Build a student and a teacher

In [ ]:
# A CNN student: frozen trunk + a generatable fc head.
student = make_cnn_student("A", feature_dim=FEATURE_DIM, num_classes=NUM_CLASSES, in_ch=3)

# The teacher writes the fc head's {weight, bias}. sigma_pi=False = the plain additive teacher.
teacher = FCMapTeacher(NUM_CLASSES, FEATURE_DIM, proto_channels=4, width=64, sigma_pi=False)

def build_prototype(student, batch):
    x, y = batch
    return feature_class_stats(student.extract_features(x), y, NUM_CLASSES)

def forward(student, batch, gen):
    x, y = batch
    return student(x, generated_fc=gen), y

## Synthetic data

Random 32x32 images and labels - enough to exercise the full API without any download.

In [ ]:
def rand_batch(n=64):
    return (torch.randn(n, 3, 32, 32), torch.randint(0, NUM_CLASSES, (n,)))

support_batches = [rand_batch() for _ in range(3)]
eval_batches = [rand_batch() for _ in range(4)]

## Leg 1 - zero-shot weight generation

Generate an fc head from the support set (no gradient steps) and evaluate, against the
nearest-centroid (NCC) and random baselines.

In [ ]:
gen = generate_averaged(teacher, student, support_batches, build_prototype)
acc = evaluate_accuracy(student, eval_batches, gen, forward)

sx, sy = support_batches[0]
centroids = class_centroids(student.extract_features(sx), sy, NUM_CLASSES)
ncc = evaluate_accuracy(student, eval_batches, centroids_to_fc(centroids), forward)
rand = evaluate_accuracy(student, eval_batches, random_like(gen), forward)

print(f"generated={acc:.3f}  ncc={ncc:.3f}  random={rand:.3f}"
      "   (~chance: untrained teacher, random data)")

## Leg 2 - warm start (recovery curve)

Install the generated head, then fine-tune it, tracking accuracy vs step.

In [ ]:
model = copy.deepcopy(student)
with torch.no_grad():
    model.fc.weight.copy_(gen["weight"])
    model.fc.bias.copy_(gen["bias"])

curve = recovery_curve(
    model,
    init=lambda m: [m.fc.weight, m.fc.bias],
    sample_batch=rand_batch,
    forward=forward,
    eval_fn=lambda m: evaluate_accuracy(m, eval_batches, None, forward),
    steps=60, lr=1e-3, eval_every=20,
)
print("(step, accuracy):", [(s, round(a, 3)) for s, a in curve])

## Leg 3 - a cheap ensemble

Generate several heads from different support draws and combine them.

In [ ]:
members = []
for support in support_batches:
    g = generate_averaged(teacher, student, [support], build_prototype)
    logits = torch.cat([forward(student, b, g)[0] for b in eval_batches])
    members.append(logits.softmax(dim=-1))
probs = torch.stack(members)                       # [M, N, C]
labels = torch.cat([b[1] for b in eval_batches])

print("ensemble acc :", round(ensemble_accuracy(probs, labels), 3))
print("ensemble gain:", round(ensemble_gain(probs, labels), 3))
print("diversity    :", round(pairwise_disagreement(probs), 3))

> **Full run.** For the real CIFAR-10 version - a trained teacher over a student
> population, with meaningful zero-shot / warm-start / ensemble numbers - run
> `python -m polyweave.experiments.cifar_fc`.